In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn

print("Python environment:", sys.executable)
print("All required packages imported successfully.")

In [ ]:
from pathlib import Path

# Find the data folder whether the notebook runs from
# the project folder or the notebooks folder.
data_path = Path("data/wdbc.data")

if not data_path.exists():
    data_path = Path("../data/wdbc.data")

measurements = [
    "radius", "texture", "perimeter", "area", "smoothness",
    "compactness", "concavity", "concave_points",
    "symmetry", "fractal_dimension"
]

feature_names = [
    f"{measurement}_{summary}"
    for summary in ["mean", "se", "worst"]
    for measurement in measurements
]

columns = ["id", "diagnosis"] + feature_names

df = pd.read_csv(data_path, header=None, names=columns)

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Diagnosis counts:")
print(df["diagnosis"].value_counts())

print("\nTotal missing values:")
print(df.isna().sum().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nDuplicate IDs:")
print(df["id"].duplicated().sum())

print("\nNon-numeric measurement columns:")
print(df[feature_names].select_dtypes(exclude="number").columns.tolist())

In [ ]:
from sklearn.model_selection import train_test_split

# Use only the 30 measurements as model inputs.
X = df[feature_names].copy()

# Encode benign as 0 and malignant as 1.
y = df["diagnosis"].map({"B": 0, "M": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

print("\nTraining diagnosis counts:")
print(y_train.value_counts().rename(index={0: "Benign", 1: "Malignant"}))

print("\nTest diagnosis counts:")
print(y_test.value_counts().rename(index={0: "Benign", 1: "Malignant"}))

In [ ]:
training_counts = (
    y_train.value_counts()
    .reindex([0, 1])
    .rename(index={0: "Benign", 1: "Malignant"})
)

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(
    training_counts.index,
    training_counts.values,
    color=["#2A9D8F", "#E76F51"]
)

ax.bar_label(bars, padding=3)
ax.set_title("Diagnosis Distribution — Training Set")
ax.set_xlabel("Diagnosis")
ax.set_ylabel("Number of samples")
ax.set_ylim(0, training_counts.max() * 1.15)

plt.tight_layout()
plt.show()

print("Training class percentages:")
print((training_counts / training_counts.sum() * 100).round(1))

In [ ]:
# Align each training sample with its diagnosis.
train_df = X_train.copy()
train_df["Diagnosis"] = y_train.map({0: "Benign", 1: "Malignant"})

features_to_plot = [
    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean"
]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, feature in zip(axes.flat, features_to_plot):
    sns.boxplot(
        data=train_df,
        x="Diagnosis",
        y=feature,
        order=["Benign", "Malignant"],
        ax=ax
    )
    ax.set_title(feature.replace("_", " ").title())
    ax.set_xlabel("")

fig.suptitle("Cell Measurements by Diagnosis — Training Set")
plt.tight_layout()
plt.show()

In [ ]:
mean_features = [
    feature for feature in feature_names
    if feature.endswith("_mean")
]

correlation_matrix = X_train[mean_features].corr()

plt.figure(figsize=(11, 8))

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    vmin=-1,
    vmax=1,
    center=0,
    square=True
)

plt.title("Correlations Between Mean Measurements — Training Set")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, random_state=42))
    ]),

    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(kernel="rbf", random_state=42))
    ]),

    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            n_jobs=-1
        ))
    ])
}

for name in models:
    print(f"{name}: pipeline created")

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, recall_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "sensitivity": make_scorer(recall_score, pos_label=1),
    "specificity": make_scorer(recall_score, pos_label=0),
    "precision": make_scorer(
        __import__("sklearn.metrics", fromlist=["precision_score"]).precision_score,
        zero_division=0
    ),
    "roc_auc": "roc_auc"
}

cv_results = {}
comparison_rows = []

for name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        error_score="raise"
    )

    cv_results[name] = scores
    row = {"Model": name}

    for metric in scoring:
        values = scores[f"test_{metric}"]
        row[f"{metric}_mean"] = values.mean()
        row[f"{metric}_std"] = values.std()

    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows).set_index("Model")

display(comparison_df.round(3))

In [ ]:
from sklearn.model_selection import cross_val_predict

oof_probabilities = cross_val_predict(
    models["Logistic Regression"],
    X_train,
    y_train,
    cv=cv,
    method="predict_proba"
)[:, 1]

print("Number of validation probabilities:", len(oof_probabilities))
print("Lowest probability:", round(oof_probabilities.min(), 4))
print("Highest probability:", round(oof_probabilities.max(), 4))

validation_predictions = pd.DataFrame({
    "Actual diagnosis": y_train.map({0: "Benign", 1: "Malignant"}),
    "Predicted malignant probability": oof_probabilities
})

display(validation_predictions.head())

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score

threshold_rows = []

for threshold in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    predictions = (oof_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_train, predictions, labels=[0, 1]
    ).ravel()

    threshold_rows.append({
        "Threshold": threshold,
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(
            y_train, predictions, zero_division=0
        ),
        "Missed malignant (FN)": fn,
        "False alarms (FP)": fp
    })

threshold_df = pd.DataFrame(threshold_rows)

display(threshold_df.round(3))

In [ ]:
from sklearn.base import clone
from sklearn.metrics import accuracy_score, roc_auc_score

final_model = clone(models["Logistic Regression"])
final_model.fit(X_train, y_train)

test_probabilities = final_model.predict_proba(X_test)[:, 1]

test_rows = []

for threshold in [0.50, 0.30]:
    predictions = (test_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()

    test_rows.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, predictions),
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Missed malignant (FN)": fn,
        "False alarms (FP)": fp
    })

test_results_df = pd.DataFrame(test_rows)

display(test_results_df.round(3))
print("Test ROC-AUC:", round(
    roc_auc_score(y_test, test_probabilities), 3
))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, threshold in zip(axes, [0.50, 0.30]):
    predictions = (test_probabilities >= threshold).astype(int)

    cm = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    )

    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Benign", "Malignant"]
    ).plot(
        ax=ax,
        cmap="Blues",
        colorbar=False,
        values_format="d"
    )

    ax.set_title(f"Test Set — Threshold {threshold:.2f}")

plt.tight_layout()
plt.show()

In [ ]:
chosen_threshold = 0.30
test_predictions = (
    test_probabilities >= chosen_threshold
).astype(int)

test_details = pd.DataFrame(index=X_test.index)

test_details["Sample ID"] = df.loc[X_test.index, "id"]
test_details["Actual diagnosis"] = y_test.map({
    0: "Benign", 1: "Malignant"
})
test_details["Predicted diagnosis"] = pd.Series(
    test_predictions,
    index=X_test.index
).map({0: "Benign", 1: "Malignant"})

test_details["Malignant probability"] = test_probabilities

misclassified_samples = test_details[
    test_details["Actual diagnosis"]
    != test_details["Predicted diagnosis"]
].copy()

print("Misclassified test samples:", len(misclassified_samples))
display(misclassified_samples.round({"Malignant probability": 4}))

In [ ]:
coefficients = pd.Series(
    final_model.named_steps["model"].coef_[0],
    index=feature_names,
    name="Coefficient"
)

top_features = coefficients.abs().nlargest(10).index
top_coefficients = coefficients.loc[top_features].sort_values()

colors = [
    "#E76F51" if value > 0 else "#2A9D8F"
    for value in top_coefficients
]

fig, ax = plt.subplots(figsize=(9, 6))

ax.barh(
    top_coefficients.index,
    top_coefficients.values,
    color=colors
)

ax.axvline(0, color="black", linewidth=1)
ax.set_title("Largest Logistic Regression Coefficients")
ax.set_xlabel("Coefficient for a standardized feature")

plt.tight_layout()
plt.show()

display(
    top_coefficients
    .sort_values(key=lambda values: values.abs(), ascending=False)
    .to_frame()
    .round(3)
)

In [ ]:
# Find the malignant test sample predicted as benign.
missed_indices = test_details.index[
    (test_details["Actual diagnosis"] == "Malignant")
    & (test_details["Predicted diagnosis"] == "Benign")
]

sample_index = missed_indices[0]
sample = X_test.loc[[sample_index]]

scaled_sample = final_model.named_steps["scaler"].transform(sample)[0]
model_coefficients = final_model.named_steps["model"].coef_[0]
intercept = final_model.named_steps["model"].intercept_[0]

contributions = pd.Series(
    scaled_sample * model_coefficients,
    index=feature_names
)

top_indices = contributions.abs().nlargest(10).index
top_contributions = contributions.loc[top_indices].sort_values()

fig, ax = plt.subplots(figsize=(9, 6))

ax.barh(
    top_contributions.index,
    top_contributions.values,
    color=[
        "#E76F51" if value > 0 else "#2A9D8F"
        for value in top_contributions
    ]
)

ax.axvline(0, color="black", linewidth=1)
ax.set_title(
    f"Feature Contributions — Missed Malignant Sample "
    f"{df.loc[sample_index, 'id']}"
)
ax.set_xlabel("Contribution to malignant log-odds")

plt.tight_layout()
plt.show()

print("Intercept:", round(intercept, 4))
print("Sum of ALL feature contributions:", round(contributions.sum(), 4))
print("Total log-odds:", round(intercept + contributions.sum(), 4))
print("Predicted malignant probability:",
      round(final_model.predict_proba(sample)[0, 1], 4))

display(top_contributions.to_frame("Contribution").round(3))

In [ ]:
from sklearn.metrics import brier_score_loss

calibration_data = pd.DataFrame({
    "probability": oof_probabilities,
    "actual": y_train.to_numpy()
})

# Group predictions into five equally sized groups.
calibration_data["group"] = pd.qcut(
    calibration_data["probability"],
    q=5,
    duplicates="drop"
)

calibration_summary = (
    calibration_data.groupby("group", observed=True)
    .agg(
        mean_probability=("probability", "mean"),
        observed_malignant_fraction=("actual", "mean"),
        sample_count=("actual", "size")
    )
    .reset_index()
)

fig, ax = plt.subplots(figsize=(6, 5))

ax.plot([0, 1], [0, 1], "--", color="gray",
        label="Perfect calibration")

ax.plot(
    calibration_summary["mean_probability"],
    calibration_summary["observed_malignant_fraction"],
    "o-",
    label="Logistic regression — OOF"
)

ax.set_xlabel("Mean predicted malignant probability")
ax.set_ylabel("Observed malignant fraction")
ax.set_title("Calibration — Training Validation Predictions")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()
plt.show()

display(calibration_summary.round(3))

print("OOF Brier score:", round(
    brier_score_loss(y_train, oof_probabilities), 4
))

In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, test_probabilities)
test_auc = roc_auc_score(y_test, test_probabilities)

fig, ax = plt.subplots(figsize=(7, 5))

ax.plot(fpr, tpr, label=f"Logistic regression — AUC {test_auc:.3f}")
ax.plot([0, 1], [0, 1], "--", color="gray",
        label="Random ranking")

for threshold, color in [(0.50, "#2A9D8F"), (0.30, "#E76F51")]:
    predictions = (test_probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()

    ax.scatter(
        fp / (fp + tn),
        tp / (tp + fn),
        color=color,
        s=80,
        label=f"Threshold {threshold:.2f}",
        zorder=3
    )

ax.set_xlabel("False-positive rate — 1 − specificity")
ax.set_ylabel("Sensitivity")
ax.set_title("ROC Curve — Held-Out Test Set")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
def wilson_interval(successes, total, z=1.96):
    proportion = successes / total
    denominator = 1 + z**2 / total

    center = (
        proportion + z**2 / (2 * total)
    ) / denominator

    margin = (
        z * np.sqrt(
            proportion * (1 - proportion) / total
            + z**2 / (4 * total**2)
        )
    ) / denominator

    return center - margin, center + margin


interval_rows = []

for threshold in [0.50, 0.30]:
    predictions = (test_probabilities >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test, predictions, labels=[0, 1]
    ).ravel()

    for metric, successes, total in [
        ("Sensitivity", tp, tp + fn),
        ("Specificity", tn, tn + fp)
    ]:
        lower, upper = wilson_interval(successes, total)

        interval_rows.append({
            "Threshold": threshold,
            "Metric": metric,
            "Correct / Total": f"{successes}/{total}",
            "Estimate": successes / total,
            "95% CI lower": lower,
            "95% CI upper": upper
        })

confidence_intervals_df = pd.DataFrame(interval_rows)
display(confidence_intervals_df.round(3))

In [ ]:
import joblib

project_root = data_path.resolve().parent.parent
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

model_bundle = {
    "model": final_model,
    "threshold": chosen_threshold,
    "comparison_threshold": 0.50,
    "feature_names": feature_names,
    "class_mapping": {0: "Benign", 1: "Malignant"},
    "random_state": 42,
    "sklearn_version": sklearn.__version__
}

joblib.dump(
    model_bundle,
    results_dir / "cell_classifier.joblib"
)

tables_to_save = {
    "cross_validation_results.csv": comparison_df,
    "validation_threshold_results.csv": threshold_df,
    "test_results.csv": test_results_df,
    "test_confidence_intervals.csv": confidence_intervals_df,
    "test_predictions.csv": test_details,
    "misclassified_test_samples.csv": misclassified_samples,
    "calibration_summary.csv": calibration_summary,
    "logistic_regression_coefficients.csv": coefficients.to_frame()
}

for filename, table in tables_to_save.items():
    table.to_csv(results_dir / filename, index=True)

print("Model and result tables saved to:")
print(results_dir)

In [ ]:
loaded_bundle = joblib.load(
    results_dir / "cell_classifier.joblib"
)

loaded_model = loaded_bundle["model"]

loaded_probabilities = loaded_model.predict_proba(
    X_test[loaded_bundle["feature_names"]]
)[:, 1]

assert np.allclose(
    loaded_probabilities,
    test_probabilities,
    rtol=1e-12,
    atol=1e-12
), "Reloaded predictions do not match."

assert loaded_bundle["threshold"] == chosen_threshold

print("Saved model loaded successfully.")
print("Predictions match the original model.")
print("Saved threshold:", loaded_bundle["threshold"])
print("Number of input features:", len(loaded_bundle["feature_names"]))

In [ ]:
validation_export = pd.DataFrame({
    "actual": y_train.to_numpy(),
    "malignant_probability": oof_probabilities
})

validation_export.to_csv(
    results_dir / "validation_predictions.csv",
    index=False
)

print("Validation predictions saved.")

## Module 1 — Results and Limitations

### Objective
Classify breast-mass samples as benign or malignant using 30 numerical
cell-nucleus measurements from the Wisconsin Diagnostic Breast Cancer dataset.

### Method
- Dataset: 569 samples, with no missing values or duplicate IDs.
- Stratified split: 455 training samples and 114 held-out test samples.
- Compared logistic regression, SVM, and random forest using five-fold
  cross-validation on the training set.
- Standardization was included inside the logistic regression and SVM pipelines.
- Selected logistic regression for the threshold experiment.
- Compared fixed thresholds of 0.50 and 0.30, using training out-of-fold
  predictions to choose 0.30 before test evaluation.

### Held-Out Test Results

| Metric | Threshold 0.50 | Threshold 0.30 |
|---|---:|---:|
| Accuracy | 96.5% | 98.2% |
| Sensitivity | 92.9% | 97.6% |
| Specificity | 98.6% | 98.6% |
| Missed malignant samples | 3 | 1 |
| False alarms | 1 | 1 |

Test ROC-AUC: 0.996.

At threshold 0.30:
- Sensitivity: 41/42; 95% Wilson CI approximately 87.7%–99.6%.
- Specificity: 71/72; 95% Wilson CI approximately 92.5%–99.8%.

### Explainability and Probability Assessment
- Examined standardized logistic regression coefficients.
- Decomposed the missed malignant sample's prediction into feature contributions.
- Training out-of-fold Brier score: 0.0191.
- A five-group calibration analysis showed broadly reasonable agreement,
  but provided only a coarse assessment.

### Limitations
- This is an educational research model, not a clinical diagnostic tool.
- The dataset describes sampled breast masses; it does not establish
  performance for screening the general population.
- Test results come from one small split of the same source dataset.
- Model and threshold selection used training validation results;
  these results may be optimistic after selection.
- External validation is needed to assess generalization.
- Coefficients and contributions explain model calculations, not biological causes.
- No model or threshold changes were made based on the test results.

### Dataset Source
https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic
